# BAN 5753 Exercise 10 — Yelp Review Synthetic Labeling and Sentiment Classification

This study uses Mistral to generate labels for 1,500 Yelp reviews, audits those labels, and fine-tunes DistilBERT to predict binary sentiment. The held-out set measures agreement with the audited synthetic labels.

**Run in Colab:** Select a GPU runtime and run the cells in order. A fresh runtime avoids previously imported package versions. The default path restores the original labeling run from the embedded CSV snapshot and needs no Google Drive mount. Set `RUN_MISTRAL = True` to rerun the four prompt examples and export a new labeling run for a separate audit. The 7B model was originally run on an NVIDIA L4 in FP16.

**Sources:** [Yelp Review Full](https://huggingface.co/datasets/Yelp/yelp_review_full), [Mistral 7B Instruct v0.1](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1), and [DistilBERT Base Uncased](https://huggingface.co/distilbert/distilbert-base-uncased). The embedded CSV contains the 1,500 selected public-dataset review texts, Yelp ratings, and original generated responses and parsed labels.

## 1. Environment and run settings

In [1]:
%pip install -q "transformers==4.55.2" "datasets==4.0.0" "accelerate==1.15.0" "scikit-learn==1.6.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
import gc
from pathlib import Path

import datasets
import sklearn
import torch
import transformers
from IPython.display import display

RUN_MISTRAL = False
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running this notebook.")

output_folder = Path.cwd() / "exercise10_outputs"
output_folder.mkdir(exist_ok=True)
snapshot_path = output_folder / "yelp_synthetic_labels.csv"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)

GPU: Tesla T4
PyTorch: 2.14.1+cu130
Transformers: 4.55.2
Datasets: 4.0.0
scikit-learn: 1.6.1


## Direction 1: Select and justify an LLM

I selected Mistral 7B Instruct v0.1 because it can follow prompts that ask it to label Yelp reviews as positive or negative. I used the same checkpoint as the course notebook and loaded it in 16-bit precision on the NVIDIA L4 GPU assigned by Colab.

In [3]:
if RUN_MISTRAL:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    # Load the same tokenizer used in the professor's notebook.
    tokenizer = AutoTokenizer.from_pretrained(
        "mistralai/Mistral-7B-Instruct-v0.1", revision="ec5deb64f2c6e6fa90c1abf74a91d5c93a9669ca"
    )

    # Load the professor's model on the L4 GPU.
    # Use 16-bit precision to reduce GPU memory usage.
    model = AutoModelForCausalLM.from_pretrained(
        "mistralai/Mistral-7B-Instruct-v0.1",
        revision="ec5deb64f2c6e6fa90c1abf74a91d5c93a9669ca",
        device_map="cuda",
        torch_dtype=torch.float16,
    )

    # Verify the GPU, device, and precision before testing prompts.
    print("GPU:", torch.cuda.get_device_name(0))
    print("Model device:", next(model.parameters()).device)
    print("Model precision:", next(model.parameters()).dtype)

## Direction 2: Design and compare two prompts

In [4]:
# Both prompts use the same labels and final output format.
# The chain-of-thought prompt adds a short explanation before the label.
simple_prompt = """Classify the overall sentiment of this hotel review.
Choose either positive or negative.
Reply with exactly one line in the format Label: <positive or negative>.

Review:
{review}
"""

cot_prompt = """Classify the overall sentiment of this hotel review.
First, identify the main praise or complaints in one short sentence.
If the review is mixed, consider which points drive the overall impression.
Then give your final answer on a separate line in the format
Label: <positive or negative>.

Review:
{review}
"""

# Expected labels are for our comparison only.
examples = [
    (
        "The room was spotless, the bed was comfortable, and I would stay here again.",
        "positive",
    ),
    (
        "The shower was broken and the front desk never sent anyone to fix it.",
        "negative",
    ),
    (
        "Check-in took a while, but the room was lovely and we had a great stay.",
        "positive",
    ),
    (
        "The location was convenient, but the room smelled like smoke and we left early.",
        "negative",
    ),
]

In [5]:
import re


def generate_response(prompt_template, review):
    # Format the review as a user message using the professor's chat template.
    prompt = prompt_template.format(review=review)
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    # Use the same generation settings for both prompts.
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the tokens generated after the prompt.
    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = output[0, prompt_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Accept positive or negative with an optional "Label:" prefix and period.
label_pattern = re.compile(
    r"(?:Label:\s*)?(positive|negative)\.?",
    re.IGNORECASE,
)


def parse_sentiment(response):
    # Read the final non-empty line so the model can explain its answer first.
    lines = [line.strip() for line in response.splitlines() if line.strip()]
    if not lines:
        return None

    match = label_pattern.fullmatch(lines[-1])
    return match.group(1).lower() if match else None

In [6]:
if RUN_MISTRAL:
    scores = {"Simple": 0, "Chain of thought": 0}

    for number, (review, expected_label) in enumerate(examples, start=1):
        print(f"\nReview {number}: {review}")
        print(f"Expected label: {expected_label}")

        for prompt_name, prompt_template in [
            ("Simple", simple_prompt),
            ("Chain of thought", cot_prompt),
        ]:
            response = generate_response(prompt_template, review)
            label = parse_sentiment(response)

            # Count a response as correct when its label matches the expected one.
            if label == expected_label:
                scores[prompt_name] += 1

            print(f"\n{prompt_name} output:\n{response}")
            print(f"Checked label: {label or 'invalid label'}")

    print("\nCorrect binary sentiment labels")
    for prompt_name, score in scores.items():
        print(f"{prompt_name}: {score}/{len(examples)}")

I tested both prompts on the same four example hotel reviews. The simple prompt produced the expected positive or negative label for all four examples. The chain-of-thought prompt correctly labeled two examples but returned “Neutral” for a review that described a great stay despite slow check-in and “Mixed” for a review where the guests left early because the room smelled like smoke. I selected the simple prompt because it consistently returned one of the two required labels in this comparison.

The scores above are the recorded results from the original run. The optional experiment can be rerun on the selected GPU.

## Direction 3: Generate labels for 1,500 Yelp reviews

The original run sampled from the train split with seed 42. The optional path pins the current repository revisions for future runs. The historical run did not record all package and model revisions. A fresh labeling run writes a separate CSV because changed responses or row ordering require a fresh audit.

In [7]:
if RUN_MISTRAL:
    from datasets import load_dataset

    # Use the assignment's Yelp dataset and a fixed seed for reproducibility.
    yelp_reviews = load_dataset(
        "Yelp/yelp_review_full", split="train",
        revision="c1f9ee939b7d05667af864ee1cb066393154bf85",
    )
    sampled_reviews = yelp_reviews.shuffle(seed=42).select(range(1500))

    import csv
    from tqdm.auto import tqdm

    output_path = output_folder / "yelp_synthetic_labels_regenerated.csv"

    # Save each review as soon as it is labeled.
    # Keep the raw response and the original Yelp rating for the label audit.
    with open(output_path, "w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(
            output_file,
            fieldnames=[
                "review_id",
                "review_text",
                "yelp_stars",
                "raw_response",
                "synthetic_label",
            ],
        )
        writer.writeheader()

        for review_id, row in enumerate(tqdm(sampled_reviews, desc="Labeling reviews")):
            # Pass only the review text to Mistral.
            response = generate_response(simple_prompt, row["text"])
            label = parse_sentiment(response)

            writer.writerow({
                "review_id": review_id,
                "review_text": row["text"],
                "yelp_stars": row["label"] + 1,
                "raw_response": response,
                "synthetic_label": label,
            })
            output_file.flush()

    print(f"Saved {len(sampled_reviews):,} reviews to {output_path}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

### Restore the original labeling data

This cell recreates the original CSV byte for byte. Gzip keeps the embedded CSV compact, Base64 stores the compressed bytes as text, and SHA-256 checks the snapshot before writing it.

In [8]:
import base64
import gzip
import hashlib

snapshot_base64 = (
    "..."
)

snapshot_bytes = gzip.decompress(base64.b64decode(snapshot_base64))
if hashlib.sha256(snapshot_bytes).hexdigest() != "0ef97e7fa449ea1146f2dfcbf8b0efaf091b85b02f7f38cc59e29260b735253f":
    raise ValueError("The embedded labeling snapshot failed its integrity check.")
snapshot_path.write_bytes(snapshot_bytes)
print("Restored original labeling snapshot:", snapshot_path)

Restored original labeling snapshot: /content/exercise10_outputs/yelp_synthetic_labels.csv


## Direction 4: Check and remove bad labels

In [9]:
import pandas as pd

labels_df = pd.read_csv(snapshot_path)

# Count reviews with a usable positive or negative label.
valid_label = labels_df["synthetic_label"].isin(["positive", "negative"])
invalid_labels = labels_df.loc[~valid_label]

print(f"Total reviews: {len(labels_df):,}")
print(f"Valid binary labels: {valid_label.sum():,}")
print(f"Missing or invalid labels: {len(invalid_labels):,}")
print("\nValid label counts")
print(labels_df.loc[valid_label, "synthetic_label"].value_counts().to_string())

# Show up to five raw responses that the parser could not label.
if not invalid_labels.empty:
    print("\nExamples of missing or invalid labels")
    display(invalid_labels[["review_id", "raw_response"]].head(5))

# Flag disagreements between Mistral's label and low or high Yelp ratings.
# Read each review before deciding whether to remove its label.
disagreement = (
    (labels_df["yelp_stars"].isin([1, 2]) & labels_df["synthetic_label"].eq("positive"))
    | (labels_df["yelp_stars"].isin([4, 5]) & labels_df["synthetic_label"].eq("negative"))
)

print(f"\nPotential disagreements to review: {disagreement.sum():,}")

# Display flagged IDs. Use review_text to revisit individual audit decisions.
display(labels_df.loc[disagreement, ["review_id", "yelp_stars", "synthetic_label"]])

Total reviews: 1,500
Valid binary labels: 1,486
Missing or invalid labels: 14

Valid label counts
synthetic_label
negative    879
positive    607

Examples of missing or invalid labels


,review_id,raw_response
14,14,Label: mixed.
36,36,Label: neutral.
49,49,Label: neutral.
172,172,Label: mixed.
552,552,Label: mixed.



Potential disagreements to review: 61


,review_id,yelp_stars,synthetic_label
6,6,4,negative
55,55,4,negative
73,73,4,negative
78,78,4,negative
79,79,2,positive
...,...,...,...
1335,1335,4,negative
1344,1344,4,negative
1375,1375,5,negative
1378,1378,4,negative


I used disagreements between Yelp ratings and Mistral’s labels to flag reviews for inspection. I read all 61 flagged reviews and removed 28 where Mistral assigned a negative label even though the review was clearly positive overall. I kept borderline cases where either label was reasonable. I also removed 14 reviews where Mistral returned mixed or neutral because the assignment uses two classes.

After removing 42 reviews, I had 1,458 reviews left, with 851 negative and 607 positive labels. The file `yelp_label_exclusion_audit.csv` records the reason for each removal.

The audit used ratings to locate disagreements and review text to decide exclusions. Review 1199 is an example of a synthetic negative label removed after the text clearly supported a positive reading. This audit is specific to the embedded responses.

In [10]:
import pandas as pd

# I removed these reviews because Mistral labeled them negative even
# though the text was clearly positive overall. I kept reviews where
# the feedback was mixed enough that either label was reasonable.
# This keeps clear labeling errors out of the training data.
manual_reasons = {
    6: "The reviewer is a regular and recommends the bar as part of a night out. "
       "They describe the jukebox issue as minor, and the restroom lines do not change their recommendation.",

    55: "The reviewer calls the restaurant a great alternative and talks about dishes to try on return visits. "
        "The negative comments apply to specific dishes, while the overall recommendation is favorable.",

    78: "The reviewer says they will definitely return, then describes a second visit favorably. "
        "The price and portion complaints qualify a positive assessment supported by both visits.",

    111: "The reviewer favors this buffet location over another and recommends its breakfast for quality and value. "
         "The grumpy closing joke is about the reviewer, so it does not support a negative label for the restaurant.",

    125: "The reviewer praises the food and service and says this restaurant delivers where other expensive brunch spots disappointed them. "
         "The harsh language near the end targets other diners and parenting, outside the assessment of the restaurant.",

    173: "The reviewer enjoys the beer, food, and roasted peanuts. "
         "The phrase 'ain't too shabby' is idiomatic praise, so treating it as a complaint reverses its meaning.",

    196: "The reviewer explicitly calls it a good experience and a great restaurant. "
         "The salmon and slow check delivery explain reservations about an otherwise favorable meal.",

    273: "The reviewer describes the hotel rooms as clean, spacious, and nice. "
         "The complaint targets conditions in the surrounding streets, while the assessment of the hotel itself is favorable.",

    405: "The reviewer opens by saying they enjoyed the show and praises the actors, audience interaction, and food. "
         "The criticism applies to the introductory scene within a favorable assessment of the show.",

    808: "The reviewer explicitly recommends the restaurant for a warm meal and says they would definitely return. "
         "The meat portions and sauce options are specific complaints within that positive conclusion.",

    828: "The reviewer says the spring rolls keep them coming back and reports enjoying them even more on a later visit. "
         "The unfavorable assessment applies mainly to one beer.",

    851: "The reviewer describes the venue as very kid-friendly and explicitly identifies summer heat and lack of air conditioning "
         "as the sole drawback. The favorable assessment applies to the venue for its intended audience.",

    863: "The reviewer visits specifically for the snacks and enthusiastically endorses them. "
         "The worn setting and concerns about some patrons are secondary to the experience they came for.",

    920: "The review endorses the core services through good prices, friendly barbers, decent haircuts, and excellent shaves. "
         "The requested changes involve the TVs and wall portraits.",

    922: "The reviewer says the coffee shop was very good. "
         "The disappointment is about losing a valued business when it closed, which reinforces the favorable assessment of the shop.",

    925: "The reviewer calls the performances great and explicitly says the show is worth the money for its audience. "
         "The mocking remarks concern individual performers' appearances and do not change the recommendation.",

    999: "The reviewer misses this shop's chai and holds it up as the standard for other versions. "
         "The exaggerated frustration comes from being unable to find a comparable drink elsewhere.",

    1043: "The reviewer explicitly calls it an excellent lunch buffet and says the restroom hand dryers cost it a star. "
          "The negative language describes that facility issue within a favorable assessment of the meal.",

    1074: "The reviewer opens with a joke about authenticity, then explicitly says they love the restaurant for its speed and convenience. "
          "Their stated overall preference is favorable.",

    1125: "The reviewer has returned for over a year and says the salon's hair care merits five stars. "
          "Price and parking reduce their rating while the evaluation of the service remains strongly positive.",

    1187: "The reviewer repeatedly visits and describes a favorite dish as a must-order every time. "
          "The complaint concerns the full-price charge for a child who barely ate, while the food receives a strong endorsement.",

    1189: "The reviewer enjoyed the exhibits and plans to return for another visit. "
          "They explain that the planetarium disappointment came from choosing a show on a different topic than they wanted.",

    1199: "The reviewer calls it a great restaurant, praises the food and wine, and has returned for lunch and dinner. "
          "They explicitly identify the silverware as the only reason they withheld five stars.",

    1250: "The reviewer calls the visit wonderful, praises the shrimp and value, and recommends several items. "
          "The negative remarks concern isolated details within an explicitly favorable assessment.",

    1267: "The reviewer calls it a great hotel, praises the suite and buffet, and explains what kept the stay from earning five stars. "
          "The limited dining choices and poor breakfast are deductions from a positive overall assessment.",

    1316: "The tire shop had the right tires, helped immediately, and charged less than the initial quote. "
          "The flat tire and impractical AAA recommendation caused the frustration, while the shop resolved the problem.",

    1375: "The German review enthusiastically endorses the pub and praises its location, atmosphere, and drink selection. "
          "Its colloquial descriptions of staying longer or returning express enthusiasm for the venue.",

    1378: "The reviewer praises the service and pizza crust and plans to return with a request for extra cheese. "
          "Their recommendation to skip the martini is specific to the drinks and leaves the favorable meal assessment intact.",
}

# Use this list in the next cleaning cell so both outputs use the same IDs.
confirmed_bad_ids = list(manual_reasons)

# Find invalid outputs and manually reviewed labeling errors in the raw data.
valid_label = labels_df["synthetic_label"].isin(["positive", "negative"])
manual_error = labels_df["review_id"].isin(confirmed_bad_ids)
audit_columns = ["review_id", "yelp_stars", "raw_response", "synthetic_label"]

# Give each invalid output its own row and record Mistral's exact response.
invalid_audit = labels_df.loc[~valid_label, audit_columns].copy()
invalid_audit["exclusion_type"] = "Invalid model output"
invalid_audit["reason"] = invalid_audit["raw_response"].fillna("").map(
    lambda response: (
        f"Mistral returned {response.strip()!r}. "
        "The two required labels are positive and negative."
    )
)

# Add the review-specific reason for each manually identified error.
manual_audit = labels_df.loc[valid_label & manual_error, audit_columns].copy()
manual_audit["exclusion_type"] = "Reviewed labeling error"
manual_audit["reason"] = manual_audit["review_id"].map(manual_reasons)

# Verify the counts and IDs before writing the CSV.
assert len(invalid_audit) == 14, "Expected 14 invalid model outputs."
assert len(manual_audit) == 28, "Expected 28 reviewed labeling errors."
assert set(manual_audit["review_id"]) == set(confirmed_bad_ids), "Check the reviewed IDs."

exclusion_audit = pd.concat([invalid_audit, manual_audit], ignore_index=True)
assert len(exclusion_audit) == 42 and exclusion_audit["review_id"].is_unique

# Put the decision and reason beside the review ID for easy scanning.
exclusion_audit = exclusion_audit[
    ["review_id", "exclusion_type", "reason", "raw_response",
     "synthetic_label", "yelp_stars"]
].sort_values(["exclusion_type", "review_id"], ignore_index=True)

# Save an Excel-compatible CSV to the study output directory.
audit_path = output_folder / "yelp_label_exclusion_audit.csv"
exclusion_audit.to_csv(audit_path, index=False, encoding="utf-8-sig")

print(f"Saved {len(exclusion_audit)} exclusion records to the study output directory.")
print(exclusion_audit["exclusion_type"].value_counts().to_string())
display(exclusion_audit.head(8))

Saved 42 exclusion records to the study output directory.
exclusion_type
Reviewed labeling error    28
Invalid model output       14


,review_id,exclusion_type,reason,raw_response,synthetic_label,yelp_stars
0,14,Invalid model output,Mistral returned 'Label: mixed.'. The two requ...,Label: mixed.,NaN,4
1,36,Invalid model output,Mistral returned 'Label: neutral.'. The two re...,Label: neutral.,NaN,3
2,49,Invalid model output,Mistral returned 'Label: neutral.'. The two re...,Label: neutral.,NaN,3
3,172,Invalid model output,Mistral returned 'Label: mixed.'. The two requ...,Label: mixed.,NaN,4
4,552,Invalid model output,Mistral returned 'Label: mixed.'. The two requ...,Label: mixed.,NaN,3
5,567,Invalid model output,Mistral returned 'Label: mixed.'. The two requ...,Label: mixed.,NaN,3
6,631,Invalid model output,Mistral returned 'Label: neutral.'. The two re...,Label: neutral.,NaN,2
7,687,Invalid model output,Mistral returned 'Label: neutral.'. The two re...,Label: neutral.,NaN,3


In [11]:
# Remove exactly the reviews documented in the exclusion audit.
excluded_ids = exclusion_audit["review_id"]
excluded = labels_df["review_id"].isin(excluded_ids)
cleaned_reviews = labels_df.loc[~excluded].copy()

# Verify that every exclusion matched and every retained label is binary.
assert excluded.sum() == len(excluded_ids), "Check the excluded review IDs."
assert len(cleaned_reviews) == 1458, "Expected 1,458 cleaned reviews."
assert cleaned_reviews["synthetic_label"].isin(["positive", "negative"]).all()

# Save the cleaned reviews as a separate file in the study output directory.
cleaned_path = output_folder / "yelp_synthetic_labels_cleaned.csv"
cleaned_reviews.to_csv(cleaned_path, index=False)

print(f"Original reviews: {len(labels_df):,}")
print(f"Reviews removed: {excluded.sum():,}")
print(f"Cleaned reviews: {len(cleaned_reviews):,}")
print("\nCleaned label counts")
print(cleaned_reviews["synthetic_label"].value_counts().to_string())
print(f"\nSaved to {cleaned_path}")

Original reviews: 1,500
Reviews removed: 42
Cleaned reviews: 1,458

Cleaned label counts
synthetic_label
negative    851
positive    607

Saved to /content/exercise10_outputs/yelp_synthetic_labels_cleaned.csv


## Direction 5: Create a train/test split

I used a stratified 80/20 split with a random seed of 42. This kept the proportions of positive and negative labels similar in both sets. The split produced 1,166 training reviews and 292 held-out reviews.

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Keep the review ID for verification, the text for model input,
# and Mistral's synthetic label as the target.
split_data = cleaned_reviews[
    ["review_id", "review_text", "synthetic_label"]
].copy()

# Hold out 20% of reviews and preserve the class proportions.
train_reviews, test_reviews = train_test_split(
    split_data,
    test_size=0.20,
    random_state=42,
    stratify=split_data["synthetic_label"],
)

# Verify that every review is assigned and no ID appears in both sets.
assert len(train_reviews) + len(test_reviews) == len(split_data)
assert set(train_reviews["review_id"]).isdisjoint(test_reviews["review_id"])

# Show the number of positive and negative labels in each split.
split_counts = pd.DataFrame({
    "Train": train_reviews["synthetic_label"].value_counts(),
    "Held out": test_reviews["synthetic_label"].value_counts(),
})
split_counts.loc["Total"] = split_counts.sum()
display(split_counts)

,Train,Held out
synthetic_label,,
negative,681,170
positive,485,122
Total,1166,292


## Direction 6: Select an appropriate classification model

I selected DistilBERT to classify Yelp reviews as positive or negative. It is a smaller version of BERT, which makes it practical to fine-tune multiple times in Colab while testing learning rate, batch size, and number of epochs. I chose the base checkpoint so I could fine-tune it on the cleaned Yelp reviews using the synthetic sentiment labels.

In [13]:
# Select the pretrained checkpoint for binary sentiment classification.
classifier_checkpoint = "distilbert/distilbert-base-uncased"

print(f"Selected classifier checkpoint: {classifier_checkpoint}")
classifier_revision = "12040accade4e8a0f71eabdb258fecc2e7e948be"

Selected classifier checkpoint: distilbert/distilbert-base-uncased


## Direction 7: Initialize the model with the correct class

I loaded the pretrained DistilBERT checkpoint with `AutoModelForSequenceClassification` and configured it to classify each review as negative (0) or positive (1).

In [14]:
from transformers import AutoModelForSequenceClassification

# Map the two sentiment labels to the IDs used during training.
label2id = {"negative": 0, "positive": 1}
id2label = {label_id: label for label, label_id in label2id.items()}

# Initialize DistilBERT with a two-class classification head.
classifier = AutoModelForSequenceClassification.from_pretrained(
    classifier_checkpoint,
    revision=classifier_revision,
    num_labels=2,
    label2id=label2id,
    id2label=id2label,
)

# Confirm the model class and label mapping for the assignment.
print("Model class:", type(classifier).__name__)
print("Number of labels:", classifier.config.num_labels)
print("Label mapping:", classifier.config.id2label)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model class: DistilBertForSequenceClassification
Number of labels: 2
Label mapping: {0: 'negative', 1: 'positive'}


## Direction 8: Define a compute metrics function

I used validation accuracy to compare the training runs. For each review, I selected the class with the highest model score and compared it with the synthetic label in the held-out set. Accuracy is the proportion of held-out reviews where those labels agree.

In [15]:
import numpy as np


def compute_metrics(eval_pred):
    # Unpack the model scores and the held-out label IDs.
    logits, label_ids = eval_pred

    # Select the class with the highest score for each review.
    predicted_ids = np.argmax(logits, axis=-1)

    # Return the fraction of predictions that match the held-out labels.
    accuracy = np.mean(predicted_ids == label_ids)
    return {"accuracy": float(accuracy)}


# Check the function with two examples and one correct prediction.
example_logits = np.array([[2.0, 0.5], [0.4, 1.8]])
example_labels = np.array([0, 0])

print("Accuracy check:", compute_metrics((example_logits, example_labels)))

Accuracy check: {'accuracy': 0.5}


## Direction 9: Show the effect of learning rate, batch size, and number of epochs

I used Hugging Face Trainer to run a baseline and three comparisons. I changed one hyperparameter at a time and started each run from the same pretrained DistilBERT checkpoint with the same random seed. Each run used the same 1,166 training reviews and 292 held-out reviews. Validation accuracy measures agreement with the synthetic sentiment labels on the held-out reviews.

In [16]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# Use the tokenizer that matches the pretrained DistilBERT checkpoint.
classifier_tokenizer = AutoTokenizer.from_pretrained(
    classifier_checkpoint, revision=classifier_revision
)


def prepare_reviews(reviews):
    # Convert the synthetic labels to the IDs defined in Direction 7.
    dataset = Dataset.from_dict({
        "review_text": reviews["review_text"].tolist(),
        "labels": reviews["synthetic_label"].map(label2id).astype(int).tolist(),
    })

    # Truncate longer reviews to 256 tokens for each experiment.
    return dataset.map(
        lambda batch: classifier_tokenizer(
            batch["review_text"], truncation=True, max_length=256
        ),
        batched=True,
        remove_columns=["review_text"],
    )


train_dataset = prepare_reviews(train_reviews)
validation_dataset = prepare_reviews(test_reviews)

# Pad each batch to the length of its longest review.
data_collator = DataCollatorWithPadding(tokenizer=classifier_tokenizer)

print(f"Training reviews: {len(train_dataset):,}")
print(f"Validation reviews: {len(validation_dataset):,}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1166 [00:00<?, ? examples/s]

Map:   0%|          | 0/292 [00:00<?, ? examples/s]

Training reviews: 1,166
Validation reviews: 292


In [17]:
import pandas as pd
import torch
from transformers import Trainer, TrainingArguments, set_seed

# Each comparison changes one setting from the baseline.
experiments = [
    ("Baseline",            2e-5,  8, 2),
    ("Higher learning rate", 5e-5,  8, 2),
    ("Larger batch size",    2e-5, 16, 2),
    ("More epochs",          2e-5,  8, 3),
]

results = []

for run_number, (run_name, learning_rate, batch_size, epochs) in enumerate(
    experiments, start=1
):
    print(f"\nRun {run_number} of {len(experiments)}: {run_name}")

    # Reset the seed before creating a fresh classification head.
    set_seed(42)
    run_model = AutoModelForSequenceClassification.from_pretrained(
        classifier_checkpoint,
        revision=classifier_revision,
        num_labels=2,
        label2id=label2id,
        id2label=id2label,
    )

    training_args = TrainingArguments(
        output_dir=str(output_folder / f"trainer_run_{run_number}"),
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=run_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Use the validation result logged at the final epoch.
    final_eval = [
        entry for entry in trainer.state.log_history if "eval_accuracy" in entry
    ][-1]

    results.append({
        "Run": run_name,
        "Learning rate": learning_rate,
        "Train batch size": batch_size,
        "Epochs": epochs,
        "Validation accuracy (%)": 100 * final_eval["eval_accuracy"],
    })

    # Release this run's model and Trainer before starting the next run.
    del trainer, run_model
    gc.collect()
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
baseline_accuracy = results_df.loc[0, "Validation accuracy (%)"]
results_df["Change vs. baseline (pp)"] = (
    results_df["Validation accuracy (%)"] - baseline_accuracy
)

display(results_df.round({
    "Validation accuracy (%)": 2,
    "Change vs. baseline (pp)": 2,
}))

results_df.to_csv(output_folder / "yelp_validation_results.csv", index=False)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Run 1 of 4: Baseline


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.330187,0.883562
2,No log,0.392942,0.863014


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Run 2 of 4: Higher learning rate


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.394078,0.873288
2,No log,0.408651,0.869863


... (3 outputs left out)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Run 4 of 4: More epochs


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.357825,0.873288
2,No log,0.370164,0.886986
3,No log,0.391169,0.866438


,Run,Learning rate,Train batch size,Epochs,Validation accuracy (%),Change vs. baseline (pp)
0,Baseline,0.00002,8,2,86.30,0.00
1,Higher learning rate,0.00005,8,2,86.99,0.68
2,Larger batch size,0.00002,16,2,87.67,1.37
3,More epochs,0.00002,8,3,86.64,0.34


## Recorded assignment results

I compared validation accuracy at the final epoch of each run. The baseline used a learning rate of 0.00002, a training batch size of 8, and 2 epochs. It achieved 86.30% validation accuracy. I changed one setting at a time and used the same training and validation reviews for each run.

- **Learning rate:** Increasing the learning rate from 0.00002 to 0.00005 decreased final validation accuracy from 86.30% to 85.96%, a change of −0.34 percentage points.
- **Batch size:** Increasing the batch size from 8 to 16 increased final validation accuracy from 86.30% to 87.33%, a change of +1.03 percentage points.
- **Number of epochs:** Increasing training from 2 to 3 epochs increased final validation accuracy from 86.30% to 87.67%, a change of +1.37 percentage points.

The three-epoch run had the highest final validation accuracy among the configurations I tested. The batch-size-16 run reached 88.36% after its first epoch and finished at 87.33% after its second. These scores measure agreement with Mistral's synthetic labels on the 292 held-out reviews.

| Configuration | Learning rate | Batch size | Epochs | Final validation accuracy | Change, percentage points |
|---|---:|---:|---:|---:|---:|
| Baseline | 0.00002 | 8 | 2 | 86.30% | 0.00 |
| Higher learning rate | 0.00005 | 8 | 2 | 85.96% | -0.34 |
| Larger batch size | 0.00002 | 16 | 2 | 87.33% | +1.03 |
| More epochs | 0.00002 | 8 | 3 | 87.67% | +1.37 |

These are the original Colab results. The training cell reports fresh results for the current run. GPU and package differences may change the resulting scores even with fixed seeds and identical data.

### Reusing the study

| Component | Setting to revisit |
|---|---|
| Dataset and labeling | Dataset revision, sampling seed, review field, prompt, and allowed labels |
| Audit | Review invalid outputs and disagreements before changing or reusing exclusion IDs |
| Classifier | Pretrained model, matching tokenizer, class IDs, and maximum input length |
| Evaluation | Keep the validation rows fixed for comparisons and use independently labeled data for claims about real-world accuracy |
| Artifacts | The output folder holds the original data, exclusion audit, cleaned data, and newly calculated results |